# Conservation Connections Manager

This python notebook processes, modifies, and manages data from notes.

## Load Libraries and Database Status

In [72]:
from yaml import load, dump
try:
    from yaml import CLoader as Loader, CDumper as Dumper
except ImportError:
    from yaml import Loader, Dumper
import yaml
import md2docx_python
import os
import json
from datetime import datetime

with open('status.json', 'r') as file:
    db_status = json.load(file)

last_run_date = datetime.strptime(db_status["last-run-date"], "%Y-%m-%d")


## Process File Functions

In [93]:

notes_data = {
    "header_yaml": {},
    "notes": [],
    "tasks" : [],
    "ideas": [],
    "resources": {}
    }
kb_data = {"nodes": {}, "edges":{}}

def process_yaml(json):
    # get passed json from embedded yaml, and decide what to do with it
    global kb_data, notes_data

    for k, v in json.items():
        if k in kb_data:
            kb_data[k] = kb_data[k] | v
        else:
            if type(v) == dict:
                notes_data[k] = notes_data[k] | v
            else:
                notes_data[k] = notes_data[k] + v


def retrieve_yaml(file):
    results = {}
    # go through file line by line
    yaml_ind = False
    header_yaml = True
    yaml_str = ""
    mdStr = ""

    with open(file, "r", encoding="utf-8-sig") as file:
        count = 0
        for line in file:

            # print(str(yaml_ind) + line)
            if yaml_ind & any(i in line for i in ["---", "```"]):
                yaml_ind = False

                #close yaml, process
                if header_yaml:
                    header_yaml = False
                    process_yaml(
                        {
                            "header_yaml": yaml.load(yaml_str, Loader=Loader)
                        }
                    )
                else:
                    process_yaml(yaml.load(yaml_str, Loader=Loader))
                yaml_str = ""

            elif any(i in line for i in ["---", "```yaml"]):
                yaml_ind = True

            elif yaml_ind:
                yaml_str += line
            else:
                mdStr += line
            count += 1

    print(f"{count} lines processed")
    # print(yaml_str)
    # for data in yaml.load_all(yaml_str, Loader = Loader):
    #     print(data)
    #     results.append(data)
    return results

## Get Files

Loop through notes folder, and process all files.

In [94]:

directory_path = './notes/' # Use '.' for the current directory

for filename in os.listdir(directory_path):
    full_path = os.path.join(directory_path, filename)
    # Check if it is a file before processing
    if (
        (filename not in db_status["skip-files"]) & 
        os.path.isfile(full_path)
        ):

        mtime = os.path.getmtime(full_path)

        if last_run_date < datetime.fromtimestamp(mtime):
            file_yaml_json = retrieve_yaml(full_path)
            print(json.dumps(file_yaml_json))

print(json.dumps(kb_data, indent=2))
print(json.dumps(notes_data, indent=2))


430 lines processed
{}
{
  "nodes": {
    "project-open-standards-for-the-practice-of-conservation": {
      "short-key": "posftpoc",
      "title": "Open Standards for the Practice of Conservation",
      "type": "project",
      "description": "Methodology for determining conservation solutions by analyzing the system.",
      "tags": [
        "open source",
        "results chains"
      ]
    },
    "organization-united-states-fish-and-wildlife-service": {
      "short-key": "usfws",
      "title": "United States Fish and Wildlife Service",
      "type": "organization"
    },
    "report-birds-of-conservation-concern-report-2021": {
      "title": "Birds of Conservation Concern-Report 2021",
      "type": "report",
      "properties": {
        "url": "https://www.fws.gov/sites/default/files/documents/birds-of-conservation-concern-2021.pdf",
        "has-author": "united-states-fish-and-wildlife-service"
      }
    },
    "north-american-bird-conservation-initiative-human-dimensi

# Process Captured Data

## Reconstitute Note and Explort to Docx

## Store KB data

Create classes for both nodes and entities for the knowledge base (bird_kb.py)